In [ ]:
# Wklej wynik komendy --list:
#kafka-topics.sh --list --bootstrap-server broker:9092
#transactions

In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    tx_id = f"TX{random.randint(1, 9999):04d}"
    user_id = f"u{random.randint(1, 20):02d}"
    amount = round(random.uniform(5.0, 5000.0), 2)
    store = random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"])
    category = random.choice(["elektronika", "odzież", "żywność", "książki"])
    timestamp = datetime.now().isoformat()

    return {
        "tx_id": tx_id,
        "user_id": user_id,
        "amount": amount,
        "store": store,
        "category": category,
        "timestamp": timestamp
    }

while True:
    transaction = generate_transaction()
    producer.send('transactions', transaction)
    print(transaction)
    time.sleep(1)

Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    transaction = message.value
    if transaction["amount"] > 3000:
        print(
            f"ALERT: {transaction['tx_id']} | "
            f"{transaction['amount']} PLN | "
            f"{transaction['store']} | "
            f"{transaction['category']}"
        )

%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję i dodaję risk_level...")

for message in consumer:
    transaction = message.value

    if transaction["amount"] > 3000:
        transaction["risk_level"] = "HIGH"
    elif transaction["amount"] > 1000:
        transaction["risk_level"] = "MEDIUM"
    else:
        transaction["risk_level"] = "LOW"

    print(transaction)

Writing consumer_filter.py


In [3]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    transaction = message.value
    store = transaction["store"]
    amount = transaction["amount"]

    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0) + amount
    msg_count += 1

    if msg_count % 10 == 0:
        print("\nSklep | Liczba | Suma | Średnia")
        print("-" * 40)
        for store_name in store_counts:
            count = store_counts[store_name]
            total = total_amount[store_name]
            avg = total / count
            print(f"{store_name} | {count} | {total:.2f} | {avg:.2f}")
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "sum": 0.0,
    "min": float("inf"),
    "max": float("-inf")
})
msg_count = 0
for message in consumer:
    transaction = message.value
    category = transaction["category"]
    amount = transaction["amount"]
    stats[category]["count"] += 1
    stats[category]["sum"] += amount
    stats[category]["min"] = min(stats[category]["min"], amount)
    stats[category]["max"] = max(stats[category]["max"], amount)
    msg_count += 1
    if msg_count % 10 == 0:
        print("\nKategoria | Liczba | Przychod | Min | Max")
        print("-" * 55)
        for cat, data in stats.items():
            print(
                f"{cat} | {data['count']} | {data['sum']:.2f} | "
                f"{data['min']:.2f} | {data['max']:.2f}"
            )

Writing consumer_count.py


In [ ]:
# Zad 5.2
# 1. Konsument będzie po prostu wisiał i czekał na nowe dane, nic nie wypisze. Kafka co prawda trzyma te stare wiadomości na dysku, ale domyślnie konsument czyta tylko to, co przychodzi od momentu jego odpalenia. Żeby zassał te historyczne dane (kiedy producent już nie działa), trzeba by mu było dopisać w ustawieniach parametr auto_offset_reset='earliest'.
# 2. Podzielą się robotą (load balancing). Kafka przypisze im różne partycje z tego tematu. Jest tylko jeden haczyk – jak nasz temat ma tylko 1 partycję, to i tak całą robotę zgarnie jeden konsument. Ten drugi będzie po prostu bezczynnie czekał w rezerwie, na wypadek gdyby ten pierwszy się wywalił.
# 3. Przetwarzanie bezstanowe to takie, gdzie program przerabia każdą wiadomość osobno i od razu o niej zapomina (np. nasz filtr, co tylko sprawdzał, czy kwota > 3000 i tyle). A stanowe wymaga pamięci (stanu) – konsument musi trzymać dane z poprzednich eventów, żeby np. policzyć sumę przychodów czy średnią, tak jak w tych zadaniach z rysowaniem tabelki.


In [4]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_history = defaultdict(list)

print("Wykrywacz anomalii uruchomiony. Nasłuchuję...")

for message in consumer:
    transaction = message.value
    user_id = transaction['user_id']

    current_time = datetime.fromisoformat(transaction['timestamp'])
 
    user_history[user_id].append(current_time)
    
    recent_transactions = [
        t for t in user_history[user_id] 
        if (current_time - t).total_seconds() <= 60
    ]
    
    user_history[user_id] = recent_transactions
    
    if len(recent_transactions) > 3:
         print(f"ALERT! Użytkownik {user_id} wykonał {len(recent_transactions)} transakcje w ciągu ostatnich 60 sekund!")


Writing consumer_anomaly.py
